# Additional ML Validation Experiments

## Rice Variety Classification — Cammeo vs Osmancik

### Purpose

This notebook investigates the validity of our original
machine learning decisions.

We will examine:

1. Class imbalance and the need for SMOTE.
2. Highly correlated features and multicollinearity.
3. Feature ablation experiments.
4. Performance of selected feature combinations.
5. Logistic Regression with and without SMOTE.

All experiments will use the training dataset only.
The independent test dataset will remain untouched.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

train_path = Path("rice_train.csv")

print("Training file exists:", train_path.exists())

Training file exists: True


In [2]:
train_df = pd.read_csv(train_path)

print("Training dataset shape:", train_df.shape)

display(train_df.head())

Training dataset shape: (3048, 8)


,Area,Perimeter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,Class
0,13078.0,452.990997,183.978973,91.690010,0.866963,13385.0,0.732743,Osmancik
1,15381.0,503.579010,211.883530,93.309746,0.897810,15749.0,0.740325,Cammeo
2,11280.0,420.101013,170.670364,85.497368,0.865476,11582.0,0.643175,Osmancik
3,12408.0,454.390991,191.896530,83.033478,0.901538,12664.0,0.583906,Osmancik
4,14131.0,490.700989,206.926666,88.284050,0.904420,14519.0,0.752931,Cammeo


In [3]:
class_counts = train_df["Class"].value_counts()

class_percentages = (
    train_df["Class"].value_counts(normalize=True) * 100
)

class_summary = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages.round(2)
})

display(class_summary)

,Count,Percentage
Class,,
Osmancik,1744,57.22
Cammeo,1304,42.78


In [4]:
majority_count = class_counts.max()
minority_count = class_counts.min()

imbalance_ratio = majority_count / minority_count

print(f"Majority-to-minority ratio: {imbalance_ratio:.2f}:1")

Majority-to-minority ratio: 1.34:1


### Class Imbalance Assessment

The training dataset contains 3,048 rice grain observations.

- Osmancik: 1,744 observations (57.22%)
- Cammeo: 1,304 observations (42.78%)
- Majority-to-minority ratio: 1.34:1

The dataset exhibits mild class imbalance.

Therefore, SMOTE was not applied automatically during
the original preprocessing stage.

Instead, a stratified train-test split was used to preserve
the class proportions.

However, class distribution alone cannot determine whether
SMOTE would improve classification performance.

A controlled experiment comparing Logistic Regression with
and without SMOTE will be conducted later in this notebook.

In [5]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [9]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, recall_score
import pandas as pd

In [10]:
# ============================================================
# 3. SEPARATE INPUT FEATURES AND TARGET
# ============================================================

X_train = train_df.drop(columns=["Class"])

y_train = train_df["Class"]

print("\nX_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


# ============================================================
# 4. SELECT FEATURES
# ============================================================

# Same two features selected in Person 3's notebook

selected_features = [
    "Perimeter",
    "Major_Axis_Length"
]

X_selected = X_train[selected_features]

print("\nSelected features:", selected_features)
print("Selected dataset shape:", X_selected.shape)


# ============================================================
# 5. CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# 6. MODEL WITHOUT SMOTE
# ============================================================

model_without_smote = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])


# ============================================================
# 7. MODEL WITH SMOTE
# ============================================================

# SMOTE is applied only to the training portion
# of each cross-validation fold.

model_with_smote = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("model", LogisticRegression(max_iter=1000))
])


# ============================================================
# 8. EVALUATION METRICS
# ============================================================

scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",

    "Cammeo_Recall": make_scorer(
        recall_score,
        pos_label="Cammeo"
    ),

    "Osmancik_Recall": make_scorer(
        recall_score,
        pos_label="Osmancik"
    )
}


# ============================================================
# 9. COMPARE BOTH MODELS
# ============================================================

results = []

models = {
    "Without SMOTE": model_without_smote,
    "With SMOTE": model_with_smote
}

for name, model in models.items():

    scores = cross_validate(
        model,
        X_selected,
        y_train,
        cv=cv,
        scoring=scoring,
        error_score="raise"
    )

    results.append({
        "Model": name,

        "Accuracy": scores["test_accuracy"].mean(),

        "Macro F1": scores["test_f1_macro"].mean(),

        "Balanced Accuracy":
            scores["test_balanced_accuracy"].mean(),

        "Cammeo Recall":
            scores["test_Cammeo_Recall"].mean(),

        "Osmancik Recall":
            scores["test_Osmancik_Recall"].mean()
    })


# ============================================================
# 10. DISPLAY RESULTS
# ============================================================

comparison_df = pd.DataFrame(results)

print("\nSMOTE COMPARISON RESULTS")

display(comparison_df.round(4))


# ============================================================
# 11. CALCULATE PERFORMANCE DIFFERENCES
# ============================================================

without_smote = comparison_df.iloc[0]
with_smote = comparison_df.iloc[1]

print("\nPERFORMANCE DIFFERENCE (WITH - WITHOUT SMOTE)")

for metric in [
    "Accuracy",
    "Macro F1",
    "Balanced Accuracy",
    "Cammeo Recall",
    "Osmancik Recall"
]:
    difference = with_smote[metric] - without_smote[metric]

    print(f"{metric}: {difference:+.4f}")


# ============================================================
# 12. SAVE COMPARISON TABLE
# ============================================================

comparison_df.to_csv(
    "smote_comparison_results.csv",
    index=False
)

print("\nResults saved to smote_comparison_results.csv")


X_train shape: (3048, 7)
y_train shape: (3048,)

Selected features: ['Perimeter', 'Major_Axis_Length']
Selected dataset shape: (3048, 2)

SMOTE COMPARISON RESULTS


,Model,Accuracy,Macro F1,Balanced Accuracy,Cammeo Recall,Osmancik Recall
0,Without SMOTE,0.9337,0.9323,0.9319,0.9195,0.9444
1,With SMOTE,0.9308,0.9296,0.9314,0.9356,0.9272



PERFORMANCE DIFFERENCE (WITH - WITHOUT SMOTE)
Accuracy: -0.0030
Macro F1: -0.0027
Balanced Accuracy: -0.0005
Cammeo Recall: +0.0161
Osmancik Recall: -0.0172

Results saved to smote_comparison_results.csv


### SMOTE Experiment — Results and Decision

The experiment compared Logistic Regression with and without
SMOTE using five-fold stratified cross-validation on the
training dataset.

Without SMOTE:
- Accuracy: 93.37%
- Macro F1: 93.23%
- Balanced accuracy: 93.19%
- Cammeo recall: 91.95%
- Osmancik recall: 94.44%

With SMOTE:
- Accuracy: 93.08%
- Macro F1: 92.96%
- Balanced accuracy: 93.14%
- Cammeo recall: 93.56%
- Osmancik recall: 92.72%

SMOTE improved Cammeo recall by approximately 1.61
percentage points but reduced Osmancik recall by
approximately 1.72 percentage points.

Overall accuracy and macro F1 decreased slightly,
while balanced accuracy remained nearly unchanged.

Decision:
The original Logistic Regression model without SMOTE
is retained because it provides slightly better overall
performance and the dataset exhibits only mild class
imbalance.

SMOTE remains a possible alternative if business
requirements prioritize Cammeo recall.

Limitation:
The differences are based on five-fold cross-validation.
No statistical significance test was performed, and the
independent test set was not used in this comparison.